In [51]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/lukashekiladze/house-prices-model/selector.pkl
/kaggle/input/datasets/lukashekiladze/house-prices-model/model.pkl
/kaggle/input/datasets/lukashekiladze/house-prices-model/scaler.pkl
/kaggle/input/datasets/lukashekiladze/house-prices-model/fill_values.pkl
/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [52]:
!pip install dagshub
!pip install mlflow

In [53]:
import dagshub
import mlflow

dagshub.init(repo_owner='lshek22', repo_name='House-Prices', mlflow=True)



Initialized MLflow to track repo "lshek22/House-Prices"

Repository lshek22/House-Prices initialized!

In [54]:
test_df = pd.read_csv("/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv")
sample_submission = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv')

test_ids = test_df['Id']



In [55]:
import pickle

with open('/kaggle/input/datasets/lukashekiladze/house-prices-model/model.pkl', 'rb') as f:
    model = pickle.load(f)

with open('/kaggle/input/datasets/lukashekiladze/house-prices-model/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('/kaggle/input/datasets/lukashekiladze/house-prices-model/selector.pkl', 'rb') as f:
    selector = pickle.load(f)

print("model loaded successfully")
print(f"model type: {type(model)}")

model loaded successfully
model type: <class 'sklearn.tree._classes.DecisionTreeRegressor'>


In [56]:


with open('/kaggle/input/datasets/lukashekiladze/house-prices-model/fill_values.pkl', 'rb') as f:
    fill_values = pickle.load(f)

LotFrontage_median = fill_values['LotFrontage_median']
garage_year_median = fill_values['garage_year_median']
electrical_mode    = fill_values['electrical_mode']

test_df = test_df.drop(columns=['Id', 'PoolQC', 'Fence', 'MiscFeature', 'Alley'], errors='ignore')

test_df['LotFrontage'] = test_df['LotFrontage'].fillna(LotFrontage_median)
test_df['GarageYrBlt'] = test_df['GarageYrBlt'].fillna(garage_year_median)
test_df['Electrical']  = test_df['Electrical'].fillna(electrical_mode)

num_cols = test_df.select_dtypes(include='number').columns
cat_cols = test_df.select_dtypes(include='object').columns
test_df[num_cols] = test_df[num_cols].fillna(0)
test_df[cat_cols] = test_df[cat_cols].fillna('None')

quality_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
ordinal_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 
                'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond']

for col in ordinal_cols:
    if col in test_df.columns:
        test_df[col] = test_df[col].map(quality_map).fillna(0).astype(int)

test_df = pd.get_dummies(test_df)

train_columns = scaler.feature_names_in_ 
test_df = test_df.reindex(columns=train_columns, fill_value=0)



In [57]:
test_scaled   = scaler.transform(test_df)
test_reduced  = selector.transform(test_scaled)
predictions   = model.predict(test_reduced)



In [58]:
submission = pd.DataFrame({
    'Id': sample_submission['Id'],
    'SalePrice': predictions
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print("submission.csv saved")
submission.head()

submission.csv saved


,Id,SalePrice
0,1461,129700.000000
1,1462,161000.000000
2,1463,150000.000000
3,1464,177457.089552
4,1465,213500.000000
